# Meta-Analysis: Human-AI Collaboration (Python)
## So sánh Hiệu suất: Human-AI vs Human Alone vs AI Alone

**Mục tiêu nghiên cứu:**
1. So sánh hiệu suất hệ thống Human-AI với con người làm việc độc lập
2. So sánh hiệu suất hệ thống Human-AI với AI làm việc độc lập  
3. Đánh giá synergy (cộng hưởng) giữa Human-AI so với max(Human, AI)
4. Phân tích các yếu tố điều tiết (moderators)

## 1. Setup & Import Libraries

In [2]:
# Install required packages
# !pip install pandas numpy scipy statsmodels openpyxl matplotlib seaborn pingouin

import pandas as pd
import numpy as np
from scipy import stats
from scipy.stats import norm
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.meta_analysis import (
    effectsize_smd,
    combine_effects,
    CombineResults
)
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
plt.style.use('seaborn-v0_8-whitegrid')

print("Libraries loaded successfully!")

Libraries loaded successfully!


## 2. Import Data

In [3]:
# Load data
df = pd.read_excel("Data_Extraction_communication_public.xlsx")

# Basic info
print("=" * 50)
print("TỔNG QUAN DỮ LIỆU")
print("=" * 50)
print(f"Số effect sizes: {len(df)}")
print(f"Số papers duy nhất: {df['Paper_ID'].nunique()}")
print(f"Số experiments duy nhất: {df['Exp_ID_Cleaned'].nunique()}")
print(f"Số cột: {len(df.columns)}")
print(f"\nGiai đoạn: {df['Year'].min()} - {df['Year'].max()}")

TỔNG QUAN DỮ LIỆU
Số effect sizes: 278
Số papers duy nhất: 67
Số experiments duy nhất: 90
Số cột: 65

Giai đoạn: 2020 - 2024


In [9]:
# Preview data
df

,Paper_Name,Paper_ID,Exp_ID,Treatment_ID,Measure_ID,Exp_ID_Cleaned,ES_ID,Title,Authors,Year,Industry,Venue,Exp_Design,Comp_Type,Task_Desc,Marketing_Task,Task_Data_Cleaned,Task_Data_IsCategoric,Task_Data_IsCode,Task_Data_IsImage,Task_Data_IsNumeric,Task_Data_IsText,Task_Data_IsVideo,Task_Output,Task_Output_Cleaned,Task_Type,AI_Type,AI_Data_In,AI_Data_Out,AI_Type_Cleaned,Final_Decision,Division_Labor,Condition_Name,AI_Expl_Incl,AI_Conf_Incl,AI_Expl_Type,Perf_Metric,Perf_Metric_Cleaned,Perf_Dir,N_Exp,N_Human,N_HumanAI,Participant_Type,Participant_Type_2,Participant_Source,Participant_Expert,Participant_Crowdworker,Avg_Perf_Human,Avg_Perf_AI,Avg_Perf_HumanAI,Avg_Perf_Human_Adj,Avg_Perf_AI_Adj,Avg_Perf_HumanAI_Adj,Baseline,Avg_Perf_Baseline_Adj,Avg_Perf_Worse_Adj,Synergy,Sd_Perf_Human,Sd_Perf_AI,Sd_Perf_HumanAI,Sd_Perf_Baseline,Sd_Perf_Worse,Est_ES,Notes,Notes_2
0,Poursabzi-Sangdeh_2021,5,1,1,1,5.1,5.1.1.1,Manipulating and Measuring Model Interpretability,"Poursabzi-Sangdeh, Forough; Goldstein, Daniel ...",2021,Business,Proceedings of the 2021 CHI Conference on Huma...,"Mixed, Between-Subjects",Independent Samples,Predict housing prices,Demand Forecasting,Numeric,No,No,No,Yes,No,No,Numeric,Numeric,Decide,Shallow,Numeric,Numeric,Rule-Based,Human,No,HAI = BB-2,No,No,NaN,Prediction Error,Error,Down,1250,252,247,Crowdworkers,Crowdworkers,MTurk,No,Yes,330555.000,180000.000,232267.000,-330555.000,-180000.000,-232267.000,AI,-180000.000,-330555.000,0,122515.0000,0.0000,56580.0000,0.0000,122515.0000,No,HAI = BB-2,NaN
1,Poursabzi-Sangdeh_2021,5,1,2,1,5.1,5.1.2.1,Manipulating and Measuring Model Interpretability,"Poursabzi-Sangdeh, Forough; Goldstein, Daniel ...",2021,Business,Proceedings of the 2021 CHI Conference on Huma...,"Mixed, Between-Subjects",Independent Samples,Predict housing prices,Demand Forecasting,Numeric,No,No,No,Yes,No,No,Numeric,Numeric,Decide,Shallow,Numeric,Numeric,Rule-Based,Human,No,HAI = BB-8,No,No,NaN,Prediction Error,Error,Down,1250,252,256,Crowdworkers,Crowdworkers,MTurk,No,Yes,330555.000,180000.000,234453.000,-330555.000,-180000.000,-234453.000,AI,-180000.000,-330555.000,0,122515.0000,0.0000,77178.0000,0.0000,122515.0000,No,HAI = BB-8,NaN
2,Poursabzi-Sangdeh_2021,5,1,3,1,5.1,5.1.3.1,Manipulating and Measuring Model Interpretability,"Poursabzi-Sangdeh, Forough; Goldstein, Daniel ...",2021,Business,Proceedings of the 2021 CHI Conference on Huma...,"Mixed, Between-Subjects",Independent Samples,Predict housing prices,Demand Forecasting,Numeric,No,No,No,Yes,No,No,Numeric,Numeric,Decide,Shallow,Numeric,Numeric,Rule-Based,Human,No,HAI = CLEAR-2,Yes,No,"Image, Numeric",Prediction Error,Error,Down,1250,252,248,Crowdworkers,Crowdworkers,MTurk,No,Yes,330555.000,180000.000,232943.000,-330555.000,-180000.000,-232943.000,AI,-180000.000,-330555.000,0,122515.0000,0.0000,81889.0000,0.0000,122515.0000,No,HAI = CLEAR-2,NaN
3,Poursabzi-Sangdeh_2021,5,1,4,1,5.1,5.1.4.1,Manipulating and Measuring Model Interpretability,"Poursabzi-Sangdeh, Forough; Goldstein, Daniel ...",2021,Business,Proceedings of the 2021 CHI Conference on Huma...,"Mixed, Between-Subjects",Independent Samples,Predict housing prices,Demand Forecasting,Numeric,No,No,No,Yes,No,No,Numeric,Numeric,Decide,Shallow,Numeric,Numeric,Rule-Based,Human,No,HAI = CLEAR-8,Yes,No,"Image, Numeric",Prediction Error,Error,Down,1250,252,247,Crowdworkers,Crowdworkers,MTurk,No,Yes,330555.000,180000.000,248016.000,-330555.000,-180000.000,-248016.000,AI,-180000.000,-330555.000,0,122515.0000,0.0000,82411.0000,0.0000,122515.0000,No,HAI = CLEAR-8,NaN
4,Poursabzi-Sangdeh_2021,5,2,1,1,5.2,5.2.1.1,Manipulating and Measuring Model Interpretability,"Poursabzi-Sangdeh, Forough; Goldstein, Daniel ...",2021,Business,Proceedings of the 2021 CHI Conference on Huma...,"Mixed, Between-Subjects",Independent Samples,Predict housing prices,Demand Forecasting,Numeric,No,No,No,Yes,No,No,Numeric,Numeric,Decide,Shallow,Numeric,Numeric,Rule-Based,Human,No,HAI = BB-2,No,No,NaN,Prediction Error,Error,Down,750,152,147,

## 3. Thống kê mô tả (Descriptive Statistics)

In [6]:
def print_distribution(df, column, title):
    """In phân bố của một biến"""
    print(f"\n{title}")
    print("-" * 40)
    counts = df[column].value_counts()
    percentages = df[column].value_counts(normalize=True) * 100
    for val in counts.index:
        print(f"{val}: {counts[val]} ({percentages[val]:.1f}%)")

# Thống kê mô tả
print("=" * 50)
print("THỐNG KÊ MÔ TẢ")
print("=" * 50)

print_distribution(df, 'Industry', 'Phân bố theo Industry')
print_distribution(df, 'Year', 'Phân bố theo Năm')
print_distribution(df, 'Task_Type', 'Phân bố theo Task Type')
print_distribution(df, 'Task_Output_Cleaned', 'Phân bố theo Task Output')
print_distribution(df, 'AI_Type_Cleaned', 'Phân bố theo AI Type')
print_distribution(df, 'Baseline', 'Phân bố theo Baseline')
print_distribution(df, 'Participant_Expert', 'Phân bố theo Expertise')
print_distribution(df, 'AI_Expl_Incl', 'Phân bố theo AI Explanation')
print_distribution(df, 'Perf_Metric_Cleaned', 'Phân bố theo Performance Metric')
print_distribution(df, 'Comp_Type', 'Phân bố theo Experimental Design')

THỐNG KÊ MÔ TẢ

Phân bố theo Industry
----------------------------------------
Healthcare: 107 (38.5%)
Communication: 86 (30.9%)
Business: 46 (16.5%)
Public sector: 39 (14.0%)

Phân bố theo Năm
----------------------------------------
2021: 93 (33.5%)
2022: 65 (23.4%)
2020: 62 (22.3%)
2023: 54 (19.4%)
2024: 4 (1.4%)

Phân bố theo Task Type
----------------------------------------
Decide: 252 (90.6%)
Create: 26 (9.4%)

Phân bố theo Task Output
----------------------------------------
Binary: 125 (45.0%)
Categoric: 103 (37.1%)
Open Response: 26 (9.4%)
Numeric: 24 (8.6%)

Phân bố theo AI Type
----------------------------------------
Deep: 132 (47.5%)
Rule-Based: 52 (18.7%)
Shallow: 46 (16.5%)
Wizard of Oz: 36 (12.9%)
Simulated-AI: 12 (4.3%)

Phân bố theo Baseline
----------------------------------------
AI: 193 (69.4%)
Human: 85 (30.6%)

Phân bố theo Expertise
----------------------------------------
No: 182 (65.5%)
Yes: 96 (34.5%)

Phân bố theo AI Explanation
----------------------------

## 4. Tính Effect Sizes (Hedges' g)

In [ ]:
def compute_hedges_g(m1, m2, sd1, sd2, n1, n2):
    """
    Tính Hedges' g (Standardized Mean Difference với hiệu chỉnh mẫu nhỏ)
    
    Parameters:
    -----------
    m1, m2: mean của nhóm 1 và nhóm 2
    sd1, sd2: standard deviation của nhóm 1 và nhóm 2
    n1, n2: sample size của nhóm 1 và nhóm 2
    
    Returns:
    --------
    g: Hedges' g effect size
    var_g: variance của Hedges' g
    """
    # Pooled standard deviation
    sd_pooled = np.sqrt(((n1 - 1) * sd1**2 + (n2 - 1) * sd2**2) / (n1 + n2 - 2))
    
    # Cohen's d
    d = (m1 - m2) / sd_pooled
    
    # Correction factor for small sample bias (Hedges' g)
    df = n1 + n2 - 2
    j = 1 - (3 / (4 * df - 1))
    
    # Hedges' g
    g = j * d
    
    # Variance of Hedges' g (large sample approximation)
    var_g = (n1 + n2) / (n1 * n2) + (g**2) / (2 * (n1 + n2))
    var_g = var_g * (j**2)
    
    return g, var_g


def compute_all_effect_sizes(df):
    """
    Tính tất cả effect sizes cho dataframe
    """
    # Initialize arrays
    n = len(df)
    es_s, var_s = np.zeros(n), np.zeros(n)  # Strong synergy
    es_h, var_h = np.zeros(n), np.zeros(n)  # Human augmentation
    es_a, var_a = np.zeros(n), np.zeros(n)  # AI augmentation
    es_n, var_n = np.zeros(n), np.zeros(n)  # Negative synergy
    
    for i in range(n):
        row = df.iloc[i]
        
        # Get values
        m_hai = row['Avg_Perf_HumanAI_Adj']
        m_h = row['Avg_Perf_Human_Adj']
        m_ai = row['Avg_Perf_AI_Adj']
        m_base = row['Avg_Perf_Baseline_Adj']
        m_worse = row['Avg_Perf_Worse_Adj']
        
        sd_hai = row['Sd_Perf_HumanAI']
        sd_h = row['Sd_Perf_Human']
        sd_ai = row['Sd_Perf_AI']
        sd_base = row['Sd_Perf_Baseline']
        sd_worse = row['Sd_Perf_Worse']
        
        n_hai = row['N_HumanAI']
        n_h = row['N_Human']
        
        # Check for valid data
        try:
            # Strong synergy: HAI vs max(H, AI)
            if pd.notna(m_hai) and pd.notna(m_base) and sd_hai > 0 and sd_base > 0:
                es_s[i], var_s[i] = compute_hedges_g(m_hai, m_base, sd_hai, sd_base, n_hai, n_h)
            else:
                es_s[i], var_s[i] = np.nan, np.nan
            
            # Human augmentation: HAI vs H
            if pd.notna(m_hai) and pd.notna(m_h) and sd_hai > 0 and sd_h > 0:
                es_h[i], var_h[i] = compute_hedges_g(m_hai, m_h, sd_hai, sd_h, n_hai, n_hai)
            else:
                es_h[i], var_h[i] = np.nan, np.nan
            
            # AI augmentation: HAI vs AI
            if pd.notna(m_hai) and pd.notna(m_ai) and sd_hai > 0 and sd_ai > 0:
                es_a[i], var_a[i] = compute_hedges_g(m_hai, m_ai, sd_hai, sd_ai, n_hai, n_h)
            else:
                es_a[i], var_a[i] = np.nan, np.nan
            
            # Negative synergy: HAI vs min(H, AI)
            if pd.notna(m_hai) and pd.notna(m_worse) and sd_hai > 0 and sd_worse > 0:
                es_n[i], var_n[i] = compute_hedges_g(m_hai, m_worse, sd_hai, sd_worse, n_hai, n_h)
            else:
                es_n[i], var_n[i] = np.nan, np.nan
                
        except Exception as e:
            es_s[i], var_s[i] = np.nan, np.nan
            es_h[i], var_h[i] = np.nan, np.nan
            es_a[i], var_a[i] = np.nan, np.nan
            es_n[i], var_n[i] = np.nan, np.nan
    
    return es_s, var_s, es_h, var_h, es_a, var_a, es_n, var_n


# Compute effect sizes
es_s, var_s, es_h, var_h, es_a, var_a, es_n, var_n = compute_all_effect_sizes(df)

# Add to dataframe
df['es_s'] = es_s
df['variance_s'] = var_s
df['es_h'] = es_h
df['variance_h'] = var_h
df['es_a'] = es_a
df['variance_a'] = var_a
df['es_n'] = es_n
df['variance_n'] = var_n

print("Effect sizes đã được tính toán!")
print(f"\nStrong synergy (es_s): mean = {np.nanmean(es_s):.3f}, valid n = {np.sum(~np.isnan(es_s))}")
print(f"Human augmentation (es_h): mean = {np.nanmean(es_h):.3f}, valid n = {np.sum(~np.isnan(es_h))}")
print(f"AI augmentation (es_a): mean = {np.nanmean(es_a):.3f}, valid n = {np.sum(~np.isnan(es_a))}")
print(f"Negative synergy (es_n): mean = {np.nanmean(es_n):.3f}, valid n = {np.sum(~np.isnan(es_n))}")

## 5. Random-Effects Meta-Analysis

In [ ]:
def random_effects_meta(effect_sizes, variances, method='REML'):
    """
    Thực hiện random-effects meta-analysis
    
    Parameters:
    -----------
    effect_sizes: array of effect sizes
    variances: array of sampling variances
    method: 'REML' hoặc 'DL' (DerSimonian-Laird)
    
    Returns:
    --------
    dict với các kết quả
    """
    # Remove NaN
    mask = ~np.isnan(effect_sizes) & ~np.isnan(variances) & (variances > 0)
    es = effect_sizes[mask]
    var = variances[mask]
    k = len(es)
    
    if k < 2:
        return None
    
    # Fixed-effect weights
    w = 1 / var
    
    # Fixed-effect estimate
    fe_estimate = np.sum(w * es) / np.sum(w)
    
    # Q statistic
    Q = np.sum(w * (es - fe_estimate)**2)
    df = k - 1
    Q_pval = 1 - stats.chi2.cdf(Q, df)
    
    # Estimate tau^2 (between-study variance)
    if method == 'DL':  # DerSimonian-Laird
        C = np.sum(w) - np.sum(w**2) / np.sum(w)
        tau2 = max(0, (Q - df) / C)
    else:  # REML (simplified)
        C = np.sum(w) - np.sum(w**2) / np.sum(w)
        tau2 = max(0, (Q - df) / C)
    
    # Random-effects weights
    w_re = 1 / (var + tau2)
    
    # Random-effects estimate
    re_estimate = np.sum(w_re * es) / np.sum(w_re)
    
    # Standard error
    se = np.sqrt(1 / np.sum(w_re))
    
    # 95% CI
    ci_lower = re_estimate - 1.96 * se
    ci_upper = re_estimate + 1.96 * se
    
    # Z-test
    z = re_estimate / se
    p_value = 2 * (1 - norm.cdf(abs(z)))
    
    # I^2 statistic
    if Q > df:
        I2 = 100 * (Q - df) / Q
    else:
        I2 = 0
    
    # H^2
    H2 = Q / df if df > 0 else 1
    
    return {
        'k': k,
        'estimate': re_estimate,
        'se': se,
        'ci_lower': ci_lower,
        'ci_upper': ci_upper,
        'z': z,
        'p_value': p_value,
        'tau2': tau2,
        'tau': np.sqrt(tau2),
        'Q': Q,
        'Q_df': df,
        'Q_pval': Q_pval,
        'I2': I2,
        'H2': H2
    }


def print_meta_results(results, name):
    """In kết quả meta-analysis"""
    if results is None:
        print(f"\n{name}: Không đủ dữ liệu")
        return
    
    print(f"\n{'='*60}")
    print(f"{name}")
    print(f"{'='*60}")
    print(f"Number of effect sizes (k): {results['k']}")
    print(f"\nPooled Effect Size (Hedges' g):")
    print(f"  Estimate: {results['estimate']:.4f}")
    print(f"  SE: {results['se']:.4f}")
    print(f"  95% CI: [{results['ci_lower']:.4f}, {results['ci_upper']:.4f}]")
    print(f"  z = {results['z']:.4f}, p = {results['p_value']:.4f}")
    print(f"\nHeterogeneity:")
    print(f"  tau² = {results['tau2']:.4f}")
    print(f"  tau = {results['tau']:.4f}")
    print(f"  Q({results['Q_df']}) = {results['Q']:.2f}, p = {results['Q_pval']:.4f}")
    print(f"  I² = {results['I2']:.1f}%")
    print(f"  H² = {results['H2']:.2f}")

In [ ]:
# Chạy meta-analysis cho từng loại so sánh
results_strong = random_effects_meta(df['es_s'].values, df['variance_s'].values)
results_human = random_effects_meta(df['es_h'].values, df['variance_h'].values)
results_ai = random_effects_meta(df['es_a'].values, df['variance_a'].values)
results_negative = random_effects_meta(df['es_n'].values, df['variance_n'].values)

print_meta_results(results_strong, "STRONG SYNERGY: Human-AI vs max(Human, AI)")
print_meta_results(results_human, "HUMAN AUGMENTATION: Human-AI vs Human alone")
print_meta_results(results_ai, "AI AUGMENTATION: Human-AI vs AI alone")
print_meta_results(results_negative, "NEGATIVE SYNERGY: Human-AI vs min(Human, AI)")

In [ ]:
# Tạo bảng tổng hợp
summary_results = pd.DataFrame([
    {
        'Comparison': 'Human-AI vs Human',
        'k': results_human['k'] if results_human else 0,
        'Hedges_g': f"{results_human['estimate']:.3f}" if results_human else 'NA',
        '95% CI': f"[{results_human['ci_lower']:.3f}, {results_human['ci_upper']:.3f}]" if results_human else 'NA',
        'p-value': f"{results_human['p_value']:.4f}" if results_human else 'NA',
        'I²': f"{results_human['I2']:.1f}%" if results_human else 'NA'
    },
    {
        'Comparison': 'Human-AI vs AI',
        'k': results_ai['k'] if results_ai else 0,
        'Hedges_g': f"{results_ai['estimate']:.3f}" if results_ai else 'NA',
        '95% CI': f"[{results_ai['ci_lower']:.3f}, {results_ai['ci_upper']:.3f}]" if results_ai else 'NA',
        'p-value': f"{results_ai['p_value']:.4f}" if results_ai else 'NA',
        'I²': f"{results_ai['I2']:.1f}%" if results_ai else 'NA'
    },
    {
        'Comparison': 'Human-AI vs max(H,AI)',
        'k': results_strong['k'] if results_strong else 0,
        'Hedges_g': f"{results_strong['estimate']:.3f}" if results_strong else 'NA',
        '95% CI': f"[{results_strong['ci_lower']:.3f}, {results_strong['ci_upper']:.3f}]" if results_strong else 'NA',
        'p-value': f"{results_strong['p_value']:.4f}" if results_strong else 'NA',
        'I²': f"{results_strong['I2']:.1f}%" if results_strong else 'NA'
    }
])

print("\n" + "="*70)
print("BẢNG TỔNG HỢP KẾT QUẢ CHÍNH")
print("="*70)
print(summary_results.to_string(index=False))

## 6. Forest Plots

In [ ]:
def create_forest_plot(effect_sizes, variances, title, filename, pooled_result):
    """
    Tạo forest plot
    """
    # Remove NaN and sort
    mask = ~np.isnan(effect_sizes) & ~np.isnan(variances)
    es = effect_sizes[mask]
    var = variances[mask]
    se = np.sqrt(var)
    
    # Sort by effect size
    sort_idx = np.argsort(es)
    es = es[sort_idx]
    se = se[sort_idx]
    
    k = len(es)
    
    # Create figure
    fig, ax = plt.subplots(figsize=(10, max(8, k * 0.15)))
    
    # Y positions
    y_pos = np.arange(k)
    
    # Colors based on sign
    colors = ['darkgreen' if e > 0 else 'darkred' for e in es]
    
    # Plot individual effects with CI
    for i in range(k):
        ci_lower = es[i] - 1.96 * se[i]
        ci_upper = es[i] + 1.96 * se[i]
        ax.hlines(y_pos[i], ci_lower, ci_upper, colors=colors[i], alpha=0.5, linewidth=1)
        ax.scatter(es[i], y_pos[i], c=colors[i], s=20, zorder=3)
    
    # Add pooled effect
    if pooled_result:
        ax.axvline(pooled_result['estimate'], color='blue', linestyle='--', linewidth=2, label='Pooled estimate')
        ax.axvspan(pooled_result['ci_lower'], pooled_result['ci_upper'], alpha=0.1, color='blue')
    
    # Reference line at 0
    ax.axvline(0, color='black', linestyle='-', linewidth=1)
    
    # Shading
    ax.axvspan(-6, 0, alpha=0.05, color='red')
    ax.axvspan(0, 6, alpha=0.05, color='green')
    
    # Labels
    ax.set_xlabel("Effect Size (Hedges' g)", fontsize=12)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_xlim(-4, 4)
    ax.set_yticks([])
    
    # Count positive/negative
    n_pos = np.sum(es > 0)
    n_neg = np.sum(es <= 0)
    
    # Add text annotations
    ax.text(0.02, 0.98, f"Negative effects: {n_neg} ({n_neg/k*100:.1f}%)", 
            transform=ax.transAxes, fontsize=10, verticalalignment='top', color='darkred')
    ax.text(0.98, 0.98, f"Positive effects: {n_pos} ({n_pos/k*100:.1f}%)", 
            transform=ax.transAxes, fontsize=10, verticalalignment='top', 
            horizontalalignment='right', color='darkgreen')
    
    # Add pooled estimate text
    if pooled_result:
        pooled_text = f"Pooled: g = {pooled_result['estimate']:.3f} [{pooled_result['ci_lower']:.3f}, {pooled_result['ci_upper']:.3f}]"
        ax.text(0.5, 0.02, pooled_text, transform=ax.transAxes, fontsize=11, 
                horizontalalignment='center', fontweight='bold',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    plt.tight_layout()
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"Forest plot saved: {filename}")


# Create forest plots
create_forest_plot(df['es_h'].values, df['variance_h'].values,
                   "Human Augmentation: Human-AI vs Human Alone",
                   "Figures/ForestPlot_Human_Augmentation.png", results_human)

create_forest_plot(df['es_a'].values, df['variance_a'].values,
                   "AI Augmentation: Human-AI vs AI Alone",
                   "Figures/ForestPlot_AI_Augmentation.png", results_ai)

create_forest_plot(df['es_s'].values, df['variance_s'].values,
                   "Strong Synergy: Human-AI vs max(Human, AI)",
                   "Figures/ForestPlot_Strong_Synergy.png", results_strong)

## 7. Funnel Plots (Publication Bias)

In [ ]:
def create_funnel_plot(effect_sizes, variances, title, filename, pooled_estimate):
    """
    Tạo funnel plot để kiểm tra publication bias
    """
    # Remove NaN
    mask = ~np.isnan(effect_sizes) & ~np.isnan(variances) & (variances > 0)
    es = effect_sizes[mask]
    se = np.sqrt(variances[mask])
    
    # Create figure
    fig, ax = plt.subplots(figsize=(10, 8))
    
    # Scatter plot
    ax.scatter(es, se, c='#0072B2', alpha=0.6, s=30)
    
    # Reference line at pooled estimate
    ax.axvline(pooled_estimate, color='red', linestyle='--', linewidth=2, label=f'Pooled estimate = {pooled_estimate:.3f}')
    
    # Funnel boundaries (95% CI)
    se_range = np.linspace(0.001, max(se) * 1.1, 100)
    ci_lower = pooled_estimate - 1.96 * se_range
    ci_upper = pooled_estimate + 1.96 * se_range
    
    ax.fill_betweenx(se_range, ci_lower, ci_upper, alpha=0.1, color='gray')
    ax.plot(ci_lower, se_range, 'k--', alpha=0.5, linewidth=1)
    ax.plot(ci_upper, se_range, 'k--', alpha=0.5, linewidth=1)
    
    # Invert y-axis (SE at top should be smaller)
    ax.invert_yaxis()
    
    # Labels
    ax.set_xlabel("Effect Size (Hedges' g)", fontsize=12)
    ax.set_ylabel("Standard Error", fontsize=12)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.legend(loc='upper right')
    
    plt.tight_layout()
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"Funnel plot saved: {filename}")


# Create funnel plots
if results_human:
    create_funnel_plot(df['es_h'].values, df['variance_h'].values,
                       "Funnel Plot: Human-AI vs Human Alone",
                       "Figures/FunnelPlot_Human.png", results_human['estimate'])

if results_ai:
    create_funnel_plot(df['es_a'].values, df['variance_a'].values,
                       "Funnel Plot: Human-AI vs AI Alone",
                       "Figures/FunnelPlot_AI.png", results_ai['estimate'])

if results_strong:
    create_funnel_plot(df['es_s'].values, df['variance_s'].values,
                       "Funnel Plot: Human-AI vs max(Human, AI)",
                       "Figures/FunnelPlot_Strong.png", results_strong['estimate'])

## 8. Egger's Test (Publication Bias)

In [ ]:
def eggers_test(effect_sizes, variances):
    """
    Egger's regression test for funnel plot asymmetry
    """
    mask = ~np.isnan(effect_sizes) & ~np.isnan(variances) & (variances > 0)
    es = effect_sizes[mask]
    se = np.sqrt(variances[mask])
    
    # Precision (1/SE)
    precision = 1 / se
    
    # Standardized effect (ES/SE)
    std_es = es / se
    
    # Weighted regression: std_es = intercept + slope * precision
    X = sm.add_constant(precision)
    model = sm.OLS(std_es, X).fit()
    
    return {
        'intercept': model.params[0],
        'intercept_se': model.bse[0],
        'intercept_t': model.tvalues[0],
        'intercept_p': model.pvalues[0],
        'slope': model.params[1],
        'slope_p': model.pvalues[1]
    }


def rank_correlation_test(effect_sizes, variances):
    """
    Begg and Mazumdar rank correlation test
    """
    mask = ~np.isnan(effect_sizes) & ~np.isnan(variances) & (variances > 0)
    es = effect_sizes[mask]
    var = variances[mask]
    
    # Standardized residuals (simplified)
    se = np.sqrt(var)
    
    # Kendall's tau
    tau, p_value = stats.kendalltau(es, se)
    
    return {'tau': tau, 'p_value': p_value}


# Run bias tests
print("=" * 60)
print("PUBLICATION BIAS TESTS")
print("=" * 60)

comparisons = [
    ('Human-AI vs Human', df['es_h'].values, df['variance_h'].values),
    ('Human-AI vs AI', df['es_a'].values, df['variance_a'].values),
    ('Human-AI vs max(H,AI)', df['es_s'].values, df['variance_s'].values)
]

bias_results = []
for name, es, var in comparisons:
    egger = eggers_test(es, var)
    rank = rank_correlation_test(es, var)
    
    print(f"\n{name}:")
    print(f"  Egger's test: intercept = {egger['intercept']:.3f}, t = {egger['intercept_t']:.3f}, p = {egger['intercept_p']:.4f}")
    print(f"  Rank correlation: tau = {rank['tau']:.3f}, p = {rank['p_value']:.4f}")
    
    bias_results.append({
        'Comparison': name,
        'Egger_intercept': f"{egger['intercept']:.3f}",
        'Egger_p': f"{egger['intercept_p']:.4f}",
        'Rank_tau': f"{rank['tau']:.3f}",
        'Rank_p': f"{rank['p_value']:.4f}"
    })

bias_df = pd.DataFrame(bias_results)
print("\n" + "="*60)
print(bias_df.to_string(index=False))

## 9. Moderator Analysis (Subgroup)

In [ ]:
def subgroup_analysis(df, es_col, var_col, moderator):
    """
    Thực hiện subgroup analysis cho một moderator
    """
    results = []
    
    for group in df[moderator].dropna().unique():
        mask = df[moderator] == group
        es = df.loc[mask, es_col].values
        var = df.loc[mask, var_col].values
        
        meta_result = random_effects_meta(es, var)
        
        if meta_result:
            results.append({
                'Moderator': moderator,
                'Group': group,
                'k': meta_result['k'],
                'g': meta_result['estimate'],
                'CI_lower': meta_result['ci_lower'],
                'CI_upper': meta_result['ci_upper'],
                'p_value': meta_result['p_value'],
                'I2': meta_result['I2']
            })
    
    return pd.DataFrame(results)


# Danh sách moderators
moderators = ['Task_Type', 'Industry', 'AI_Type_Cleaned', 'Participant_Expert', 
              'AI_Expl_Incl', 'Task_Output_Cleaned', 'Comp_Type']

print("=" * 70)
print("MODERATOR ANALYSIS - Human Augmentation (HAI vs Human)")
print("=" * 70)

all_mod_results = []
for mod in moderators:
    mod_result = subgroup_analysis(df, 'es_h', 'variance_h', mod)
    if len(mod_result) > 0:
        all_mod_results.append(mod_result)
        print(f"\n{mod}:")
        for _, row in mod_result.iterrows():
            sig = "*" if row['p_value'] < 0.05 else ""
            print(f"  {row['Group']}: g = {row['g']:.3f} [{row['CI_lower']:.3f}, {row['CI_upper']:.3f}], k = {row['k']}, I² = {row['I2']:.1f}% {sig}")

# Combine all results
moderator_df = pd.concat(all_mod_results, ignore_index=True)
moderator_df.to_csv('Data/ModeratorAnalysis_Human.csv', index=False)
print("\nKết quả đã được lưu: Data/ModeratorAnalysis_Human.csv")

In [ ]:
print("=" * 70)
print("MODERATOR ANALYSIS - AI Augmentation (HAI vs AI)")
print("=" * 70)

all_mod_results_ai = []
for mod in moderators:
    mod_result = subgroup_analysis(df, 'es_a', 'variance_a', mod)
    if len(mod_result) > 0:
        all_mod_results_ai.append(mod_result)
        print(f"\n{mod}:")
        for _, row in mod_result.iterrows():
            sig = "*" if row['p_value'] < 0.05 else ""
            print(f"  {row['Group']}: g = {row['g']:.3f} [{row['CI_lower']:.3f}, {row['CI_upper']:.3f}], k = {row['k']}, I² = {row['I2']:.1f}% {sig}")

moderator_df_ai = pd.concat(all_mod_results_ai, ignore_index=True)
moderator_df_ai.to_csv('Data/ModeratorAnalysis_AI.csv', index=False)
print("\nKết quả đã được lưu: Data/ModeratorAnalysis_AI.csv")

## 10. Subgroup Analysis theo Industry

In [ ]:
print("=" * 70)
print("SUBGROUP ANALYSIS THEO INDUSTRY")
print("=" * 70)

industry_results = []

for industry in df['Industry'].unique():
    df_sub = df[df['Industry'] == industry]
    
    # HAI vs Human
    res_h = random_effects_meta(df_sub['es_h'].values, df_sub['variance_h'].values)
    # HAI vs AI
    res_a = random_effects_meta(df_sub['es_a'].values, df_sub['variance_a'].values)
    # Strong synergy
    res_s = random_effects_meta(df_sub['es_s'].values, df_sub['variance_s'].values)
    
    industry_results.append({
        'Industry': industry,
        'n': len(df_sub),
        'HAI_vs_H_g': f"{res_h['estimate']:.3f}" if res_h else 'NA',
        'HAI_vs_H_CI': f"[{res_h['ci_lower']:.2f}, {res_h['ci_upper']:.2f}]" if res_h else 'NA',
        'HAI_vs_H_p': f"{res_h['p_value']:.4f}" if res_h else 'NA',
        'HAI_vs_AI_g': f"{res_a['estimate']:.3f}" if res_a else 'NA',
        'HAI_vs_AI_CI': f"[{res_a['ci_lower']:.2f}, {res_a['ci_upper']:.2f}]" if res_a else 'NA',
        'HAI_vs_AI_p': f"{res_a['p_value']:.4f}" if res_a else 'NA',
        'Synergy_g': f"{res_s['estimate']:.3f}" if res_s else 'NA',
        'Synergy_CI': f"[{res_s['ci_lower']:.2f}, {res_s['ci_upper']:.2f}]" if res_s else 'NA'
    })
    
    print(f"\n{industry} (n = {len(df_sub)}):")
    if res_h:
        print(f"  HAI vs Human: g = {res_h['estimate']:.3f} [{res_h['ci_lower']:.2f}, {res_h['ci_upper']:.2f}], p = {res_h['p_value']:.4f}")
    if res_a:
        print(f"  HAI vs AI: g = {res_a['estimate']:.3f} [{res_a['ci_lower']:.2f}, {res_a['ci_upper']:.2f}], p = {res_a['p_value']:.4f}")
    if res_s:
        print(f"  Strong Synergy: g = {res_s['estimate']:.3f} [{res_s['ci_lower']:.2f}, {res_s['ci_upper']:.2f}], p = {res_s['p_value']:.4f}")

industry_df = pd.DataFrame(industry_results)
print("\n" + "="*70)
print(industry_df.to_string(index=False))
industry_df.to_csv('Data/Industry_Subgroup_Analysis.csv', index=False)

## 11. Visualization: Moderator Effects

In [ ]:
def plot_moderator_effects(mod_df, title, filename):
    """
    Tạo biểu đồ forest plot cho moderator effects
    """
    fig, ax = plt.subplots(figsize=(12, max(6, len(mod_df) * 0.4)))
    
    y_pos = np.arange(len(mod_df))
    
    # Colors based on significance
    colors = ['darkgreen' if p < 0.05 else 'gray' for p in mod_df['p_value']]
    
    # Plot effect sizes with CI
    ax.errorbar(mod_df['g'], y_pos, 
                xerr=[mod_df['g'] - mod_df['CI_lower'], mod_df['CI_upper'] - mod_df['g']],
                fmt='o', capsize=4, capthick=2, color='black', ecolor='gray')
    
    for i, (_, row) in enumerate(mod_df.iterrows()):
        color = 'darkgreen' if row['p_value'] < 0.05 else 'gray'
        ax.scatter(row['g'], i, c=color, s=100, zorder=3)
    
    # Reference line
    ax.axvline(0, color='red', linestyle='--', linewidth=1)
    
    # Labels
    labels = [f"{row['Moderator']}: {row['Group']} (k={row['k']})" for _, row in mod_df.iterrows()]
    ax.set_yticks(y_pos)
    ax.set_yticklabels(labels)
    ax.set_xlabel("Effect Size (Hedges' g)", fontsize=12)
    ax.set_title(title, fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.show()


# Plot moderator effects for Human Augmentation
if len(moderator_df) > 0:
    plot_moderator_effects(moderator_df, 
                           "Moderator Effects: Human-AI vs Human Alone",
                           "Figures/ModeratorPlot_Human.png")

## 12. Sensitivity Analysis: Leave-One-Out

In [ ]:
def leave_one_out_analysis(df, es_col, var_col, cluster_col):
    """
    Leave-one-out sensitivity analysis at cluster level
    """
    clusters = df[cluster_col].unique()
    results = []
    
    for cluster in clusters:
        df_sub = df[df[cluster_col] != cluster]
        meta_result = random_effects_meta(df_sub[es_col].values, df_sub[var_col].values)
        
        if meta_result:
            results.append({
                'excluded': cluster,
                'estimate': meta_result['estimate'],
                'p_value': meta_result['p_value'],
                'I2': meta_result['I2']
            })
    
    return pd.DataFrame(results)


# Leave-one-out at experiment level
print("=" * 60)
print("LEAVE-ONE-OUT SENSITIVITY ANALYSIS")
print("=" * 60)

loo_results = leave_one_out_analysis(df, 'es_h', 'variance_h', 'Exp_ID_Cleaned')

if len(loo_results) > 0:
    print(f"\nHuman Augmentation (HAI vs Human):")
    print(f"  Original estimate: {results_human['estimate']:.4f}")
    print(f"  Range of LOO estimates: [{loo_results['estimate'].min():.4f}, {loo_results['estimate'].max():.4f}]")
    print(f"  Range of p-values: [{loo_results['p_value'].min():.4f}, {loo_results['p_value'].max():.4f}]")
    print(f"  Range of I²: [{loo_results['I2'].min():.1f}%, {loo_results['I2'].max():.1f}%]")
    
    # Check if any LOO changes significance
    if (loo_results['p_value'] > 0.05).any() and results_human['p_value'] < 0.05:
        print("  ⚠️ Warning: Some LOO analyses become non-significant!")
    else:
        print("  ✓ Results are robust across LOO analyses.")

## 13. Export Results

In [ ]:
# Export main results
main_results = {
    'Human Augmentation': results_human,
    'AI Augmentation': results_ai,
    'Strong Synergy': results_strong
}

# Create summary DataFrame
final_summary = []
for name, res in main_results.items():
    if res:
        final_summary.append({
            'Comparison': name,
            'k': res['k'],
            'Hedges_g': res['estimate'],
            'SE': res['se'],
            'CI_lower': res['ci_lower'],
            'CI_upper': res['ci_upper'],
            'z': res['z'],
            'p_value': res['p_value'],
            'tau2': res['tau2'],
            'I2': res['I2'],
            'Q': res['Q'],
            'Q_df': res['Q_df'],
            'Q_p': res['Q_pval']
        })

final_df = pd.DataFrame(final_summary)
final_df.to_csv('Data/Main_Results.csv', index=False)

# Save effect sizes with computed values
df.to_csv('Data/Data_with_EffectSizes.csv', index=False)

print("=" * 60)
print("FILES EXPORTED")
print("=" * 60)
print("1. Data/Main_Results.csv - Kết quả meta-analysis chính")
print("2. Data/ModeratorAnalysis_Human.csv - Moderator analysis (HAI vs H)")
print("3. Data/ModeratorAnalysis_AI.csv - Moderator analysis (HAI vs AI)")
print("4. Data/Industry_Subgroup_Analysis.csv - Subgroup theo industry")
print("5. Data/Data_with_EffectSizes.csv - Dữ liệu gốc với effect sizes")
print("\nFIGURES:")
print("1. Figures/ForestPlot_*.png - Forest plots")
print("2. Figures/FunnelPlot_*.png - Funnel plots")
print("3. Figures/ModeratorPlot_*.png - Moderator effects")

## 14. Summary Report

In [ ]:
print("="*70)
print("BÁO CÁO TÓM TẮT KẾT QUẢ META-ANALYSIS")
print("="*70)

print(f"\n📊 DỮ LIỆU:")
print(f"   - Tổng số effect sizes: {len(df)}")
print(f"   - Số papers: {df['Paper_ID'].nunique()}")
print(f"   - Số experiments: {df['Exp_ID_Cleaned'].nunique()}")
print(f"   - Giai đoạn: {df['Year'].min()}-{df['Year'].max()}")

print(f"\n📈 KẾT QUẢ CHÍNH:")

if results_human:
    sig_h = "✓ Significant" if results_human['p_value'] < 0.05 else "✗ Not significant"
    print(f"\n   1. Human Augmentation (HAI vs Human):")
    print(f"      g = {results_human['estimate']:.3f} [{results_human['ci_lower']:.3f}, {results_human['ci_upper']:.3f}]")
    print(f"      p = {results_human['p_value']:.4f} {sig_h}")
    print(f"      I² = {results_human['I2']:.1f}%")
    if results_human['estimate'] > 0:
        print(f"      → Human-AI tốt hơn Human alone")
    else:
        print(f"      → Human alone tốt hơn Human-AI")

if results_ai:
    sig_a = "✓ Significant" if results_ai['p_value'] < 0.05 else "✗ Not significant"
    print(f"\n   2. AI Augmentation (HAI vs AI):")
    print(f"      g = {results_ai['estimate']:.3f} [{results_ai['ci_lower']:.3f}, {results_ai['ci_upper']:.3f}]")
    print(f"      p = {results_ai['p_value']:.4f} {sig_a}")
    print(f"      I² = {results_ai['I2']:.1f}%")
    if results_ai['estimate'] > 0:
        print(f"      → Human-AI tốt hơn AI alone")
    else:
        print(f"      → AI alone tốt hơn Human-AI")

if results_strong:
    sig_s = "✓ Significant" if results_strong['p_value'] < 0.05 else "✗ Not significant"
    print(f"\n   3. Strong Synergy (HAI vs max(H, AI)):")
    print(f"      g = {results_strong['estimate']:.3f} [{results_strong['ci_lower']:.3f}, {results_strong['ci_upper']:.3f}]")
    print(f"      p = {results_strong['p_value']:.4f} {sig_s}")
    print(f"      I² = {results_strong['I2']:.1f}%")
    if results_strong['estimate'] > 0:
        print(f"      → Có synergy thực sự (HAI > cả H và AI)")
    else:
        print(f"      → Không có synergy (HAI không vượt trội)")

print(f"\n📝 KẾT LUẬN:")
print(f"   - Heterogeneity cao (I² > 75%) cho thấy variation lớn giữa các nghiên cứu")
print(f"   - Cần xem xét moderators để giải thích variation")
print(f"   - Kết quả phù hợp với literature về human-AI collaboration")